# Reproducing the v2 results

This notebook re-derives every headline number in the paper from the
released splits and reports **PASS/FAIL** against the published values.

The published values below are injected from the run artifacts by
`build_repro_notebook.py`; none of them is typed by hand. That matters: the
defect this paper corrects is partly a bookkeeping one — v1's numbers drifted
between the paper, the README and the product page because each copy was
maintained separately.

**What is reproduced.** The classical models are retrained from the recorded
hyperparameters and re-evaluated on the test split at the thresholds selected
on validation. The Optuna search itself (50 trials x 5 folds) is not re-run
here — its chosen parameters are published and used directly.

**What is not.** The 10-configuration Conv-Transformer
ablation needs a GPU and hours. This notebook carries the per-class scores
from that run and recomputes the paired statistics from them, which is the
part most worth checking independently.

**Ablation coverage.** all 10 of 10 planned deep configurations completed and are checked below.

In [ ]:
import hashlib, json, os, time
from pathlib import Path
import numpy as np, pandas as pd

PUBLISHED = json.loads(r'''{"manifest": {"seed": 376, "n_features": 67, "n_train": 89973, "n_val": 11247, "n_test": 11247, "sha256_train": "3a93e58df54e989ec0d18fb480cc47b71a0d9a7ef8d762935594d560f413bcae", "sha256_val": "125937bc29ef9658c58dece44a29a576e80f37081f26625fdfd10d1e1f288176", "sha256_test": "e7478bec88008dfbe90dd5025a44b17f7ee313d9e0568683b73ad8b7a00c65c8", "positive_rate": 0.6900601954351054, "positive_rate_train": 0.6900625743278539, "positive_rate_val": 0.6900506801813817, "positive_rate_test": 0.6900506801813817, "dropped_bytecode_dups": 858, "dropped_feature_dups": 3372}, "binary": {"LogReg": {"f1": 0.8900421271139874, "f1_lo": 0.8862076886415385, "f1_hi": 0.8938863202491917, "precision": 0.845787885820376, "recall": 0.9391830949619895, "fnr": 0.060816905038010564, "mcc": 0.6096976089194128, "pr_auc": 0.9528629235504111}, "RF": {"f1": 0.945214185839583, "f1_lo": 0.9416015874497582, "f1_hi": 0.9486776216756808, "precision": 0.9326476859400477, "recall": 0.9581239530988275, "fnr": 0.04187604690117253, "mcc": 0.8185988738534258, "pr_auc": 0.9893864036401853}, "XGB": {"f1": 0.9492472845074001, "f1_lo": 0.9459716421655001, "f1_hi": 0.9525146333360236, "precision": 0.9361062390378351, "recall": 0.9627625306017266, "fnr": 0.037237469398273416, "mcc": 0.8318817791258651, "pr_auc": 0.9898970691623216}, "CatBoost": {"f1": 0.9349948612538541, "f1_lo": 0.930892048875568, "f1_hi": 0.9385318602506685, "precision": 0.9322402971692071, "recall": 0.9377657518361036, "fnr": 0.0622342481638964, "mcc": 0.7889192457128236, "pr_auc": 0.9853065639397318}}, "thresholds": {"LogReg": 0.22999999999999998, "RF": 0.5049999999999999, "XGB": 0.43499999999999994, "CatBoost": 0.3499999999999999}, "xgb_params": {"n_estimators": 1033, "max_depth": 12, "learning_rate": 0.05520690824812322, "subsample": 0.9744252195668149, "colsample_bytree": 0.6684520574911611, "scale_pos_weight": 1.3008716264941094, "min_child_weight": 1}, "multilabel": {"ML0_LogReg_balanced": 0.45381786875865193, "ML1_RandomForest": 0.6907647950533917, "ML2_XGBoost": 0.7646477442708204}, "multilabel_per_label": {"ML0_LogReg_balanced": [0.47184052641765045, 0.46757679180887374, 0.1971153846153846, 0.03224181360201511, 0.28767471410419315, 0.6191008098296565, 0.7556349642660802, 0.7993579454253612], "ML1_RandomForest": [0.7549378200438918, 0.746268656716418, 0.4842105263157895, 0.26666666666666666, 0.7038269550748752, 0.80849532037437, 0.8587477283991409, 0.902964686835981], "ML2_XGBoost": [0.800405953991881, 0.7743865948533812, 0.6666666666666666, 0.5, 0.7856598016781083, 0.8156385751520417, 0.8638959390862944, 0.9105284227381906]}, "dl": {"A1_baseline": [0.6930887699127197, 0.6635358929634094, 0.48832273483276367, 0.17777778208255768, 0.6844320297241211, 0.71285480260849, 0.7994964122772217, 0.8665735721588135], "B1_pos_weight": [0.6990768909454346, 0.6612523198127747, 0.5461121201515198, 0.3448276221752167, 0.6725185513496399, 0.7233772277832031, 0.8017216324806213, 0.8639855980873108], "B2_focal_g2": [0.6876952052116394, 0.6629087328910828, 0.5384615659713745, 0.1428571343421936, 0.6652542352676392, 0.7064953446388245, 0.7992411851882935, 0.8675681352615356], "B3_focal_pos_weight": [0.6898428201675415, 0.6520535349845886, 0.5148515105247498, 0.2978723347187042, 0.681753933429718, 0.7111568450927734, 0.792235791683197, 0.8570296764373779], "B4_asymmetric": [0.6762430667877197, 0.6365412473678589, 0.46026095747947693, 0.27848100662231445, 0.6086956858634949, 0.7015278339385986, 0.7883445024490356, 0.8633754253387451], "B5_threshold_tuning_on_best": [0.6892759203910828, 0.6452088356018066, 0.5760286450386047, 0.3103448152542114, 0.7014925479888916, 0.7144642472267151, 0.797452986240387, 0.8587626814842224], "C1_transformer_4layers": [0.6877192854881287, 0.6539633870124817, 0.5945016145706177, 0.35087722539901733, 0.685393214225769, 0.7205357551574707, 0.8023987412452698, 0.8605205416679382], "C2_dmodel_256": [0.7042345404624939, 0.6721218824386597, 0.5719626545906067, 0.38596493005752563, 0.7010014057159424, 0.7171941995620728, 0.8113646507263184, 0.8707292675971985], "C3_pure_cnn": [0.6969324946403503, 0.6346305012702942, 0.51408451795578, 0.4067797064781189, 0.6757493019104004, 0.7167291045188904, 0.7996429800987244, 0.8671521544456482], "C4_pure_transformer": [0.6534653306007385, 0.6143849492073059, 0.551928699016571, 0.21686746180057526, 0.5888198614120483, 0.6903778314590454, 0.8004627227783203, 0.8560258150100708]}, "dl_macro": {"A1_baseline": 0.6357601881027222, "B1_pos_weight": 0.6641089916229248, "B2_focal_g2": 0.6338101923465729, "B3_focal_pos_weight": 0.6495995558798313, "B4_asymmetric": 0.6266837157309055, "B5_threshold_tuning_on_best": 0.6616288349032402, "C1_transformer_4layers": 0.6694887205958366, "C2_dmodel_256": 0.6793216913938522, "C3_pure_cnn": 0.6639626026153564, "C4_pure_transformer": 0.6215415596961975}, "delta": {"delta_f1_xgb_minus_rf": 0.004072358991133093, "ci95": [0.0018443102579745712, 0.006250849163967375], "p_delta_gt_0": 1.0, "B": 2000}, "comparisons": {"A1_baseline": {"dl_macro": 0.6357602495700121, "xgb_macro": 0.7646477442708204, "gap_pp": 12.888749470080828, "xgb_wins_of_8": 8, "p_sign": 0.0078125, "p_wilcoxon": 0.0078125, "holm_alpha": 0.005, "significant_after_holm": false}, "B1_pos_weight": {"dl_macro": 0.6641089953482151, "xgb_macro": 0.7646477442708204, "gap_pp": 10.053874892260527, "xgb_wins_of_8": 8, "p_sign": 0.0078125, "p_wilcoxon": 0.0078125, "holm_alpha": 0.005555555555555556, "significant_after_holm": false}, "B2_focal_g2": {"dl_macro": 0.6338101923465729, "xgb_macro": 0.7646477442708204, "gap_pp": 13.08375519242475, "xgb_wins_of_8": 8, "p_sign": 0.0078125, "p_wilcoxon": 0.0078125, "holm_alpha": 0.00625, "significant_after_holm": false}, "B3_focal_pos_weight": {"dl_macro": 0.6495995558798313, "xgb_macro": 0.7646477442708204, "gap_pp": 11.504818839098906, "xgb_wins_of_8": 8, "p_sign": 0.0078125, "p_wilcoxon": 0.0078125, "holm_alpha": 0.0071428571428571435, "significant_after_holm": false}, "B4_asymmetric": {"dl_macro": 0.6266837157309055, "xgb_macro": 0.7646477442708204, "gap_pp": 13.796402853991484, "xgb_wins_of_8": 8, "p_sign": 0.0078125, "p_wilcoxon": 0.0078125, "holm_alpha": 0.008333333333333333, "significant_after_holm": false}, "B5_threshold_tuning_on_best": {"dl_macro": 0.6616288349032402, "xgb_macro": 0.7646477442708204, "gap_pp": 10.301890936758017, "xgb_wins_of_8": 8, "p_sign": 0.0078125, "p_wilcoxon": 0.0078125, "holm_alpha": 0.01, "significant_after_holm": false}, "C1_transformer_4layers": {"dl_macro": 0.6694887205958366, "xgb_macro": 0.7646477442708204, "gap_pp": 9.515902367498374, "xgb_wins_of_8": 8, "p_sign": 0.0078125, "p_wilcoxon": 0.0078125, "holm_alpha": 0.0125, "significant_after_holm": false}, "C2_dmodel_256": {"dl_macro": 0.6793216913938522, "xgb_macro": 0.7646477442708204, "gap_pp": 8.532605287696814, "xgb_wins_of_8": 8, "p_sign": 0.0078125, "p_wilcoxon": 0.0078125, "holm_alpha": 0.016666666666666666, "significant_after_holm": false}, "C3_pure_cnn": {"dl_macro": 0.6639625951647758, "xgb_macro": 0.7646477442708204, "gap_pp": 10.068514910604453, "xgb_wins_of_8": 8, "p_sign": 0.0078125, "p_wilcoxon": 0.0078125, "holm_alpha": 0.025, "significant_after_holm": false}, "C4_pure_transformer": {"dl_macro": 0.6215415839105844, "xgb_macro": 0.7646477442708204, "gap_pp": 14.310616036023593, "xgb_wins_of_8": 8, "p_sign": 0.0078125, "p_wilcoxon": 0.0078125, "holm_alpha": 0.05, "significant_after_holm": false}}, "labels": ["access-control", "arithmetic", "bad-randomness", "double-spending", "locked-ether", "other", "reentrancy", "unchecked-calls"], "features": ["total_instructions", "unique_instructions", "block_dependent_count", "block_dependency_index", "has_TIMESTAMP", "has_NUMBER", "has_DIFFICULTY", "has_GASLIMIT", "has_COINBASE", "has_BLOCKHASH", "environmental_instructions_count", "environmental_ratio", "unique_environmental_ops", "environmental_complexity", "balance_operations", "address_operations", "caller_operations", "origin_operations", "callvalue_operations", "external_dependency_index", "calldata_size_ops", "calldata_load_ops", "calldata_copy_ops", "total_calldata_ops", "calldata_density", "external_call_count", "has_external_calls", "call_value_ops", "call_gas_limit_ops", "potential_reentrancy_pattern", "pushes", "pops", "stack_imbalance", "stack_operations_ratio", "stack_underflow_risk", "total_gas_cost", "avg_gas_per_instruction", "max_gas_instruction", "high_gas_instructions", "gas_dos_risk_index", "arithmetic_ops_count", "arithmetic_density", "unsafe_arithmetic_pattern", "control_flow_ops", "jumpi_count", "conditional_branching_ratio", "control_flow_complexity", "caller_based_checks", "origin_usage", "access_control_ratio", "uses_origin_instead_caller", "balance_before_external_call", "randomness_ops_count", "has_bad_randomness_pattern", "dangerous_ops_count", "dangerous_ops_density", "opcode_entropy", "reentrancy_risk_score", "frontrunning_risk_score", "dos_risk_score", "arithmetic_risk_score", "overall_security_risk_score", "has_reentrancy_indicators", "has_unchecked_external_calls", "has_arithmetic_vulnerabilities", "has_access_control_issues", "has_dos_vulnerabilities"], "env": {"scikit-learn": "1.8.0", "xgboost": "3.1.2", "numpy": "1.26.4", "pandas": "2.3.3", "scipy": "1.16.3", "catboost": "1.2.8", "python": "3.12.8"}, "n_checks": 114}''')
LABELS   = PUBLISHED["labels"]
FEATURES = PUBLISHED["features"]
RESULTS  = {}          # check name -> (ok, published, reproduced)

def check(name, published, got, tol=5e-4):
    ok = (published == got) if isinstance(published, str) \
         else abs(float(published) - float(got)) <= tol
    RESULTS[name] = (ok, published, got)
    print(f"  {'PASS' if ok else 'FAIL'}  {name}: published={published} got={got}")
    return ok

def mount():
    """Kaggle has moved dataset mount points; accept either layout."""
    for c in [Path("/kaggle/input/defi-bytecode-features-v2"),
              Path("/kaggle/input/datasets/sergeisolovyev/defi-bytecode-features-v2"),
              Path(".")]:
        if (c / "train_v2.parquet").exists():
            return c
    raise SystemExit("dataset not mounted")

D = mount()
print("dataset at", D)

In [ ]:
import importlib, platform
print("library versions -- 'authored' is what produced the published "
      "numbers; a mismatch is the first thing to suspect if a check misses "
      "by a hair")
for pkg, ver in sorted(PUBLISHED["env"].items()):
    if pkg == "python":
        here = platform.python_version()
    else:
        try:
            here = importlib.import_module(
                "sklearn" if pkg == "scikit-learn" else pkg).__version__
        except ImportError:
            here = "MISSING"
    print(f"  {pkg:14s} authored={ver:10s} here={here}"
          f"{'' if here == ver else '   <-- differs'}")

## 1. Split integrity

Before any metric is worth reading, the splits must be the ones the paper
used. The digests below are the build-time SHA-256 of each parquet file, and
the no-overlap assertion is the invariant the build script enforced: no
feature vector may appear in more than one split. That invariant *is* the
correction — v1 deduplicated on the raw bytecode string, which let contracts
that are bit-identical in feature space sit on both sides of the split.

In [ ]:
def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

print("digests")
for split in ["train", "val", "test"]:
    check(f"sha256_{split}", PUBLISHED["manifest"][f"sha256_{split}"],
          sha256(D / f"{split}_v2.parquet"))

# Read only the numeric columns. The bytecode column dominates the file
# (train_v2.parquet is ~493 MB, almost all of it strings) and the classical
# pipeline never touches it -- loading it costs minutes and gigabytes for
# nothing.
COLS = FEATURES + LABELS + ["is_vulnerable"]
parts = {s: pd.read_parquet(D / f"{s}_v2.parquet", columns=COLS) for s in
         ["train", "val", "test"]}

print("\nrow counts and positive rates")
for s, df in parts.items():
    check(f"n_{s}", PUBLISHED["manifest"][f"n_{s}"], len(df))
    check(f"positive_rate_{s}", PUBLISHED["manifest"][f"positive_rate_{s}"],
          float(df["is_vulnerable"].mean()), tol=1e-6)

print("\nno feature vector appears in more than one split")
# Hash rows rather than building sets of 67-tuples: the tuple form needs
# ~90k x 67 Python floats per split and takes minutes. A 64-bit row hash
# over 112k rows has a collision probability around 3e-10, which is far
# below anything that would change the verdict.
keys = {s: set(pd.util.hash_pandas_object(df[FEATURES], index=False))
        for s, df in parts.items()}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    n = len(keys[a] & keys[b])
    RESULTS[f"overlap_{a}_{b}"] = (n == 0, 0, n)
    print(f"  {'PASS' if n == 0 else 'FAIL'}  overlap {a}/{b}: {n}")

## 2. Binary detection

Retrained from the published hyperparameters, scored on the test split at
the thresholds chosen on validation. The test split is read once, here.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             matthews_corrcoef, average_precision_score)
import xgboost as xgb

Xtr = parts["train"][FEATURES].to_numpy("float32")
ytr = parts["train"]["is_vulnerable"].to_numpy()
Xte = parts["test"][FEATURES].to_numpy("float32")
yte = parts["test"]["is_vulnerable"].to_numpy()

sc = StandardScaler().fit(Xtr)

models = {
    "XGB": xgb.XGBClassifier(**PUBLISHED["xgb_params"], n_jobs=-1,
                             random_state=42, eval_metric="logloss"),
    "RF":  RandomForestClassifier(n_estimators=500, class_weight="balanced",
                                  n_jobs=-1, random_state=42),
    "LogReg": LogisticRegression(max_iter=2000, class_weight="balanced",
                                 random_state=42),
}
SKIPPED = 0
try:
    from catboost import CatBoostClassifier
    models["CatBoost"] = CatBoostClassifier(
        iterations=800, auto_class_weights="Balanced",
        random_seed=42, verbose=False)
except ImportError:
    SKIPPED += 8
    print("catboost unavailable: the 8 published CatBoost numbers are NOT "
          "reproduced in this run")

def boot_ci(y, s, thr, B=1000, seed=42):
    """Stratified percentile bootstrap for F1 -- ported verbatim from
    run_classical_v2.py:114-122, so the published interval is recomputed
    rather than restated."""
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    vals = []
    for _ in range(B):
        i = np.concatenate([rng.choice(pos, len(pos)),
                            rng.choice(neg, len(neg))])
        vals.append(f1_score(y[i], (s[i] >= thr).astype(int)))
    return float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975))

probs = {}

for name, m in models.items():
    t0 = time.time()
    A, B = (sc.transform(Xtr), sc.transform(Xte)) if name == "LogReg" \
           else (Xtr, Xte)
    m.fit(A, ytr)
    p = m.predict_proba(B)[:, 1]
    probs[name] = p
    pred = (p >= PUBLISHED["thresholds"][name]).astype(int)
    got = {"f1": f1_score(yte, pred), "precision": precision_score(yte, pred),
           "recall": recall_score(yte, pred), "mcc": matthews_corrcoef(yte, pred),
           "pr_auc": average_precision_score(yte, p)}
    got["fnr"] = 1 - got["recall"]
    got["f1_lo"], got["f1_hi"] = boot_ci(yte, p,
                                         PUBLISHED["thresholds"][name])
    print(f"\n{name}  ({time.time()-t0:.0f}s)")
    # Tolerances calibrated on two MEASURED signatures from real runs:
    #  * correct spec across library builds (authored: sklearn 1.8.0 /
    #    xgboost 3.1.2; Kaggle: newer): F1 agrees to 4e-6, while
    #    precision/recall/FNR/MCC drift up to 1.6e-3 -- a different build
    #    lands a handful of borderline contracts across the threshold;
    #  * a WRONG spec (RF as 400 unweighted trees instead of 500 balanced)
    #    moved those components by 3.0e-3..4.3e-3 and was caught.
    # Headline scores stay tight; components get 2e-3, which separates
    # version noise from a genuine specification error.
    for k in ["f1", "pr_auc"]:
        check(f"binary_{name}_{k}", PUBLISHED["binary"][name][k], got[k],
              tol=5e-4)
    for k in ["precision", "recall", "fnr", "mcc"]:
        check(f"binary_{name}_{k}", PUBLISHED["binary"][name][k], got[k],
              tol=2e-3)
    # looser: these two also carry the resampler's stream. Measured spread
    # across rng seeds is <= 2.4e-4.
    for k in ["f1_lo", "f1_hi"]:
        check(f"binary_{name}_{k}", PUBLISHED["binary"][name][k], got[k],
              tol=1e-3)

## 2b. Is XGBoost really ahead of the random forest?

Not an argument from the two intervals above: overlapping marginal CIs do
not license a difference claim, and leaning on them was one of v1's defects.
The paper's claim rests on a *paired* bootstrap over the same resampled test
sets, recomputed here from the two score vectors section 2 just produced.

In [ ]:
rng = np.random.default_rng(42)
pos, neg = np.where(yte == 1)[0], np.where(yte == 0)[0]
deltas = []
for _ in range(PUBLISHED["delta"]["B"]):
    i = np.concatenate([rng.choice(pos, len(pos)),
                        rng.choice(neg, len(neg))])
    deltas.append(
        f1_score(yte[i], (probs["XGB"][i]
                          >= PUBLISHED["thresholds"]["XGB"]).astype(int))
        - f1_score(yte[i], (probs["RF"][i]
                            >= PUBLISHED["thresholds"]["RF"]).astype(int)))
d = np.array(deltas)
lo, hi = float(np.quantile(d, 0.025)), float(np.quantile(d, 0.975))
print(f"paired dF1 (XGB-RF) = {d.mean():+.5f}  "
      f"CI95 [{lo:+.5f}, {hi:+.5f}]  P(d>0) = {(d > 0).mean():.3f}")
check("delta_f1_xgb_minus_rf", PUBLISHED["delta"]["delta_f1_xgb_minus_rf"],
      float(d.mean()), tol=5e-4)
check("delta_ci95_lo", PUBLISHED["delta"]["ci95"][0], lo, tol=1e-3)
check("delta_ci95_hi", PUBLISHED["delta"]["ci95"][1], hi, tol=1e-3)

## 3. Multi-label

Eight one-vs-rest heads at the default threshold, exactly as in the paper --
all three estimators, including the balanced logistic regression whose
0.4538 is the leftmost column of the published per-label heatmap.

In [ ]:
from sklearn.multioutput import MultiOutputClassifier

Ytr = parts["train"][LABELS].to_numpy("int8")
Yte = parts["test"][LABELS].to_numpy("int8")

ml_models = {
    "ML2_XGBoost": MultiOutputClassifier(
        xgb.XGBClassifier(n_estimators=400, max_depth=8, n_jobs=-1,
                          random_state=42, eval_metric="logloss"), n_jobs=1),
    "ML1_RandomForest": RandomForestClassifier(
        n_estimators=400, class_weight="balanced", n_jobs=-1, random_state=42),
    # analyze_v2.py:63-64. NOT wrapped in a scaler: the multi-label LogReg
    # is fitted on the raw features, unlike the binary one. Standardising
    # here would silently produce something other than 0.4538.
    "ML0_LogReg_balanced": MultiOutputClassifier(
        LogisticRegression(max_iter=1500, class_weight="balanced"), n_jobs=-1),
}

ml_got = {}
for name, m in ml_models.items():
    t0 = time.time()
    m.fit(Xtr, Ytr)
    P = m.predict(Xte)
    per = [float(f1_score(Yte[:, i], P[:, i])) for i in range(len(LABELS))]
    ml_got[name] = per
    print(f"\n{name}  ({time.time()-t0:.0f}s)")
    check(f"multilabel_{name}", PUBLISHED["multilabel"][name],
          float(np.mean(per)), tol=3e-3)
    # the published figure is the per-label matrix, not the macro: a macro
    # check alone lets per-class errors cancel, and the rare classes
    # (ML1 double-spending 0.267, bad-randomness 0.484) are exactly where a
    # reproduction would drift.
    # 1e-2, not 5e-3: rare-class cells are decided by a few dozen
    # positives, and a different sklearn build breaks ties differently --
    # measured 7.9e-3 on ML1/bad-randomness for a spec-identical forest,
    # while a wrong forest spec shifts rare-class F1 by several times that.
    for i, lab in enumerate(LABELS):
        check(f"multilabel_{name}_{lab}",
              PUBLISHED["multilabel_per_label"][name][i], per[i], tol=1e-2)

## 4. The deep comparator, and what eight paired observations support

The ablation is not retrained here. What *is* recomputed is the statistical
claim built on it — and that claim is deliberately modest. With eight
classes the smallest two-sided p a sign test can return is 2/256 = 0.0078,
so a clean sweep means "eight out of eight" and nothing stronger. The sign
test and Wilcoxon applied to the same eight numbers are one piece of
evidence reported twice, not two confirmations; only the sign test is used.

In [ ]:
from scipy.stats import binomtest

# The comparator must be the vector section 3 just REPRODUCED. Reading the
# published one here makes the table self-referential: it would print the
# same 8/8, the same p and the same 8.53 pp even if the retrain above had
# produced nonsense.
xgbv = np.array(ml_got["ML2_XGBoost"])
rows, pvals = [], []
for cfg, per in sorted(PUBLISHED["dl"].items()):
    c = np.array(per)
    wins = int((xgbv > c).sum())
    p = binomtest(wins, len(LABELS), 0.5, alternative="two-sided").pvalue
    rows.append((cfg, float(c.mean()), wins, float(p)))
    pvals.append((cfg, float(p)))

# Holm STEP-DOWN. Sort ascending, compare p_(i) to alpha/(m-i), and stop at
# the first failure -- everything after it is retained too. Testing each p
# against its own threshold independently (what stood here) lets later
# hypotheses pass on the looser thresholds that only become available once
# the stricter ones have been rejected. With ten identical p = 0.0078125
# that marked six of ten significant; the correct answer is none, because
# the smallest p already exceeds alpha/10 = 0.005.
m = len(pvals)
holm, _rejecting = {}, True
for _r, (_k, _p) in enumerate(sorted(pvals, key=lambda t: t[1])):
    if _rejecting and _p > 0.05 / (m - _r):
        _rejecting = False
    holm[_k] = _rejecting

print(f"{'config':32s} {'macro-F1':>9s} {'XGB wins':>9s} {'p_sign':>8s}  Holm")
for cfg, macro, wins, p in sorted(rows, key=lambda t: -t[1]):
    print(f"{cfg:32s} {macro:9.4f} {wins:6d}/8 {p:8.4f}  "
          f"{'yes' if holm[cfg] else 'no'}")
    pub = PUBLISHED["comparisons"][cfg]
    check(f"dl_wins_{cfg}", pub["xgb_wins_of_8"], wins, tol=0)
    check(f"dl_psign_{cfg}", pub["p_sign"], p, tol=1e-9)
    check(f"dl_gap_pp_{cfg}", pub["gap_pp"],
          100 * (xgbv.mean() - macro), tol=0.3)
    check(f"dl_holm_{cfg}", float(pub["significant_after_holm"]),
          float(holm[cfg]), tol=0)

best = max(rows, key=lambda t: t[1])
print(f"\nbest deep configuration: {best[0]} at {best[1]:.4f}")
print(f"classical XGBoost: {xgbv.mean():.4f}")
print(f"gap: {100*(xgbv.mean()-best[1]):.2f} percentage points")
print(f"sign-test floor at n=8: {2/2**8:.4f} -- p={best[3]:.4f} IS that floor"
      if abs(best[3] - 2/2**8) < 1e-9 else "")

## 5. Verdict

In [ ]:
bad = [k for k, (ok, *_) in RESULTS.items() if not ok]
print(f"checks run: {len(RESULTS)}  (+{SKIPPED} skipped)")
print(f"expected:   {PUBLISHED['n_checks']}")
print(f"failed:     {len(bad)}")
if len(RESULTS) + SKIPPED != PUBLISHED["n_checks"]:
    raise SystemExit(
        f"COVERAGE mismatch: ran {len(RESULTS)} checks +{SKIPPED} skipped, "
        f"expected {PUBLISHED['n_checks']} -- a published number is going "
        f"unreproduced")
if bad:
    print("\nFAILED:")
    for k in bad:
        ok, pub, got = RESULTS[k]
        print(f"  {k}: published={pub} reproduced={got}")
    raise SystemExit("reproduction MISMATCH -- see above")
print("\nAll published numbers reproduced.")